# Gateway Authorization

This example proves that an autonomous Agent needs both a valid actor identity and the required subject identity before it can reach a granted resource. The executable section covers the ServiceAccount agent plane: a granted request succeeds, ungranted and malformed identities are denied, and the gateway fails closed when the policy decision point (PDP) is unavailable.

The later user-plane and gateway-bypass sections are read-only examples. They show the complete Keycloak, OIDC dynamic client registration (DCR), and Calico strict-routing flow, but Jupytext does not run them in CI.

## Agent plane: executed in CI

Install KAOS with gateway authorization backed by projected Kubernetes ServiceAccount tokens:

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT=$(git rev-parse --show-toplevel)
kaos system install --gateway-enabled --metallb-enabled \
  --agent-auth-enabled service-account \
  --chart-path "$REPO_ROOT/operator/chart" --wait

Apply the self-contained sample and wait for its workloads. The autonomous Agent is granted only `granted-model`; `unrelated-agent` exists to make the deny side visible.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT=$(git rev-parse --show-toplevel)
kubectl apply -f "$REPO_ROOT/operator/config/samples/8-access-grant.yaml"

kubectl wait --for=condition=available deployment/modelapi-granted-model \
  -n authz-demo --timeout=180s
kubectl wait --for=condition=available deployment/modelapi-ungranted-model \
  -n authz-demo --timeout=180s
kubectl wait --for=condition=available deployment/agent-autonomous-researcher \
  -n authz-demo --timeout=180s
kubectl wait --for=condition=available deployment/agent-unrelated-agent \
  -n authz-demo --timeout=180s

Mint a short-lived projected token with the gateway audience, port-forward Envoy, and assert the authorization matrix. The first request retries while the policy projection settles; every other request checks the returned status immediately.

In [ ]:
%%bash
set -euo pipefail

assert_status() {
  expected=$1
  description=$2
  shift 2
  code=$(curl -sS -o /dev/null -w '%{http_code}' "$@" || true)
  [ "$code" = "$expected" ] || {
    echo "$description: expected $expected got $code"
    exit 1
  }
  echo "$description: $code"
}

wait_for_status() {
  expected=$1
  description=$2
  shift 2
  for _ in $(seq 1 60); do
    code=$(curl -sS -o /dev/null -w '%{http_code}' "$@" || true)
    if [ "$code" = "$expected" ]; then
      echo "$description: $code"
      return 0
    fi
    sleep 2
  done
  echo "$description: expected $expected got $code"
  return 1
}

ENVOY_SERVICE=$(kubectl get service -n envoy-gateway-system \
  -l gateway.envoyproxy.io/owning-gateway-name=kaos-gateway \
  -o jsonpath='{.items[0].metadata.name}')
GATEWAY_URL=http://127.0.0.1:18888
TMP_DIR=$(git rev-parse --show-toplevel)/tmp
mkdir -p "$TMP_DIR"
ORIGINAL_PDP_REPLICAS=$(kubectl get deployment/kaos-pdp -n kaos-system \
  -o jsonpath='{.spec.replicas}')

kubectl port-forward -n envoy-gateway-system \
  "service/$ENVOY_SERVICE" 18888:80 >"$TMP_DIR/authorization-port-forward.log" 2>&1 &
PORT_FORWARD_PID=$!
cleanup() {
  kubectl scale deployment/kaos-pdp -n kaos-system \
    --replicas="${ORIGINAL_PDP_REPLICAS:-2}" >/dev/null 2>&1 || true
  kill "$PORT_FORWARD_PID" >/dev/null 2>&1 || true
}
trap cleanup EXIT

for _ in $(seq 1 30); do
  curl -s -o /dev/null "$GATEWAY_URL" && break
  sleep 1
done
kill -0 "$PORT_FORWARD_PID"

TOKEN=$(kubectl create token kaos-agent-autonomous-researcher \
  -n authz-demo --audience=kaos-gateway --duration=10m)
WRONG_AUDIENCE_TOKEN=$(kubectl create token kaos-agent-autonomous-researcher \
  -n authz-demo --audience=not-kaos-gateway --duration=10m)

GRANTED_URL="$GATEWAY_URL/authz-demo/modelapi/granted-model/health/liveliness"
UNGRANTED_URL="$GATEWAY_URL/authz-demo/modelapi/ungranted-model/health/liveliness"

wait_for_status 200 "granted resource" \
  -H "x-agent-authorization: Bearer $TOKEN" \
  -H "Authorization: Bearer $TOKEN" \
  "$GRANTED_URL"
assert_status 403 "ungranted resource" \
  -H "x-agent-authorization: Bearer $TOKEN" \
  -H "Authorization: Bearer $TOKEN" \
  "$UNGRANTED_URL"
assert_status 403 "missing subject" \
  -H "x-agent-authorization: Bearer $TOKEN" \
  "$GRANTED_URL"
assert_status 403 "wrong-audience token" \
  -H "x-agent-authorization: Bearer $WRONG_AUDIENCE_TOKEN" \
  -H "Authorization: Bearer $WRONG_AUDIENCE_TOKEN" \
  "$GRANTED_URL"

kubectl scale deployment/kaos-pdp -n kaos-system --replicas=0
kubectl wait --for=delete pod -n kaos-system \
  -l app.kubernetes.io/name=kaos-pdp --timeout=120s
wait_for_status 403 "PDP unavailable" \
  -H "x-agent-authorization: Bearer $TOKEN" \
  -H "Authorization: Bearer $TOKEN" \
  "$GRANTED_URL"

kubectl scale deployment/kaos-pdp -n kaos-system \
  --replicas="${ORIGINAL_PDP_REPLICAS:-2}"
kubectl rollout status deployment/kaos-pdp -n kaos-system --timeout=180s
wait_for_status 200 "PDP restored" \
  -H "x-agent-authorization: Bearer $TOKEN" \
  -H "Authorization: Bearer $TOKEN" \
  "$GRANTED_URL"

The important distinction is the two headers: `x-agent-authorization` identifies the calling Agent, while `Authorization` supplies the subject on whose behalf it acts. For an autonomous Agent, both can carry the same projected identity, but omitting the subject is still denied.

## Full Keycloak user and agent planes

The remaining cells are intentionally marked `.noeval`: they are rendered for readers but skipped by CI. Start with a Calico KIND cluster because Kubernetes `NetworkPolicy` objects only prove gateway-only routing when the cluster CNI enforces them.

```bash .noeval
mkdir -p tmp
cat > tmp/authz-kind.yaml <<'EOF'
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
networking:
  disableDefaultCNI: true
  podSubnet: 192.168.0.0/16
nodes:
  - role: control-plane
EOF

kind create cluster --name kaos-authz --config tmp/authz-kind.yaml
kubectl create -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.1/manifests/tigera-operator.yaml
kubectl wait --for=condition=Established crd/installations.operator.tigera.io --timeout=180s
kubectl create -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.1/manifests/custom-resources.yaml
kubectl wait --for=condition=Available tigerastatus/calico --timeout=600s
kubectl wait --for=condition=Ready node --all --timeout=300s
```

First create the grant without a user identity provider. Its `Enforced` condition is `False/NoUserIdentityProvider`; the grant cannot affect policy until the user plane exists.

```bash .noeval
set -euo pipefail
kaos system install --gateway-enabled --metallb-enabled --gateway-api-strict \
  --agent-auth-enabled service-account --user-auth-enabled none --wait
kubectl apply -f operator/config/samples/8-access-grant.yaml
kubectl run bypass-client -n authz-demo --image=curlimages/curl:8.12.1 \
  --command -- sleep infinity
kubectl wait --for=condition=Ready pod/bypass-client -n authz-demo --timeout=120s

condition=$(kubectl get accessgrant researchers-enter-autonomous -n authz-demo \
  -o jsonpath='{.status.conditions[?(@.type=="Enforced")].status}/{.status.conditions[?(@.type=="Enforced")].reason}')
[ "$condition" = "False/NoUserIdentityProvider" ] || {
  echo "expected False/NoUserIdentityProvider got $condition"
  exit 1
}
```

Upgrade to Keycloak for both planes. The operator waits for an initial-access token before it can dynamically register Agent clients, so mint that token through Keycloak's in-cluster hostname, create the bootstrap Secret, and restart the operator.

```bash .noeval
set -euo pipefail
kaos system install --gateway-enabled --metallb-enabled --gateway-api-strict \
  --agent-auth-enabled keycloak --user-auth-enabled keycloak

KC=http://keycloak.keycloak.svc.cluster.local:8080
ADMIN=$(kubectl exec -n authz-demo bypass-client -- curl -fsS -X POST \
  "$KC/realms/master/protocol/openid-connect/token" \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  --data-urlencode client_id=admin-cli \
  --data-urlencode username=admin \
  --data-urlencode password=admin \
  --data-urlencode grant_type=password | jq -r .access_token)

IAT=$(kubectl exec -n authz-demo bypass-client -- curl -fsS -X POST \
  "$KC/admin/realms/kaos/clients-initial-access" \
  -H "Authorization: Bearer $ADMIN" \
  -H 'Content-Type: application/json' \
  -d '{"expiration":86400,"count":100}' | jq -r .token)

kubectl create secret generic kaos-oidc-registration -n kaos-system \
  --from-literal=token="$IAT"
kubectl rollout restart deployment/kaos-kaos-operator-controller-manager -n kaos-system
kubectl rollout status deployment/kaos-kaos-operator-controller-manager \
  -n kaos-system --timeout=180s

for _ in $(seq 1 45); do
  condition=$(kubectl get accessgrant researchers-enter-autonomous -n authz-demo \
    -o jsonpath='{.status.conditions[?(@.type=="Enforced")].status}/{.status.conditions[?(@.type=="Enforced")].reason}')
  [ "$condition" = "True/Enforced" ] && break
  sleep 2
done
[ "$condition" = "True/Enforced" ] || {
  echo "expected True/Enforced got $condition"
  exit 1
}
```

Port-forward Envoy as in the executed section. The managed `kaos-user` belongs to `researchers`, so its user token can enter the granted Agent but not the unrelated one.

```bash .noeval
set -euo pipefail
ENVOY_SERVICE=$(kubectl get service -n envoy-gateway-system \
  -l gateway.envoyproxy.io/owning-gateway-name=kaos-gateway \
  -o jsonpath='{.items[0].metadata.name}')
kubectl port-forward -n envoy-gateway-system \
  "service/$ENVOY_SERVICE" 18888:80 >tmp/authorization-keycloak-port-forward.log 2>&1 &
GATEWAY_URL=http://127.0.0.1:18888

USER_TOKEN=$(kubectl exec -n authz-demo bypass-client -- curl -fsS -X POST \
  "$KC/realms/kaos/protocol/openid-connect/token" \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  --data-urlencode client_id=kaos \
  --data-urlencode client_secret=kaos-dev-secret \
  --data-urlencode username=kaos-user \
  --data-urlencode password=kaos-password \
  --data-urlencode grant_type=password | jq -r .access_token)

code=$(curl -sS -o /dev/null -w '%{http_code}' \
  -H "Authorization: Bearer $USER_TOKEN" \
  "$GATEWAY_URL/authz-demo/agent/autonomous-researcher/health")
[ "$code" = "200" ] || { echo "expected entry 200 got $code"; exit 1; }

code=$(curl -sS -o /dev/null -w '%{http_code}' \
  -H "Authorization: Bearer $USER_TOKEN" \
  "$GATEWAY_URL/authz-demo/agent/unrelated-agent/health")
[ "$code" = "403" ] || { echo "expected ungranted 403 got $code"; exit 1; }
```

Each Agent receives a DCR-created Secret. Exchange those client credentials for an actor token and prove the autonomous Agent still reaches only its granted ModelAPI.

```bash .noeval
set -euo pipefail
CLIENT_ID=$(kubectl get secret kaos-oidc-autonomous-researcher -n authz-demo \
  -o jsonpath='{.data.client_id}' | base64 -d)
CLIENT_SECRET=$(kubectl get secret kaos-oidc-autonomous-researcher -n authz-demo \
  -o jsonpath='{.data.client_secret}' | base64 -d)
ACTOR_TOKEN=$(kubectl exec -n authz-demo bypass-client -- curl -fsS -X POST \
  "$KC/realms/kaos/protocol/openid-connect/token" \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  --data-urlencode grant_type=client_credentials \
  --data-urlencode client_id="$CLIENT_ID" \
  --data-urlencode client_secret="$CLIENT_SECRET" | jq -r .access_token)

code=$(curl -sS -o /dev/null -w '%{http_code}' \
  -H "x-agent-authorization: Bearer $ACTOR_TOKEN" \
  -H "Authorization: Bearer $ACTOR_TOKEN" \
  "$GATEWAY_URL/authz-demo/modelapi/granted-model/health/liveliness")
[ "$code" = "200" ] || { echo "expected DCR actor 200 got $code"; exit 1; }
```

## Prove the gateway cannot be bypassed

Strict mode creates ingress policies for protected workloads and the PDP. A valid token sent directly to either ClusterIP must time out with curl exit code 28, while the same request through Envoy remains allowed.

```bash .noeval
set -euo pipefail
set +e
kubectl exec -n authz-demo bypass-client -- curl -sS -o /dev/null \
  --connect-timeout 3 --max-time 5 \
  -H "x-agent-authorization: Bearer $ACTOR_TOKEN" \
  -H "Authorization: Bearer $ACTOR_TOKEN" \
  http://modelapi-granted-model:8000/health/liveliness
modelapi_rc=$?

kubectl exec -n authz-demo bypass-client -- curl -sS -o /dev/null \
  --connect-timeout 3 --max-time 5 \
  http://kaos-pdp.kaos-system.svc:9191
pdp_rc=$?
set -e

[ "$modelapi_rc" = "28" ] || {
  echo "expected direct ModelAPI timeout (28) got $modelapi_rc"
  exit 1
}
[ "$pdp_rc" = "28" ] || {
  echo "expected direct PDP timeout (28) got $pdp_rc"
  exit 1
}

code=$(curl -sS -o /dev/null -w '%{http_code}' \
  -H "x-agent-authorization: Bearer $ACTOR_TOKEN" \
  -H "Authorization: Bearer $ACTOR_TOKEN" \
  "$GATEWAY_URL/authz-demo/modelapi/granted-model/health/liveliness")
[ "$code" = "200" ] || { echo "expected gateway 200 got $code"; exit 1; }
```

Authorization projection and mounted OPA data are eventually consistent. After a grant, issuer, or DCR mapping changes, allow up to 90 seconds for the new policy to appear before treating a denial as final.

## Manual AIB-native token-exchange runbook

This final walkthrough is `.noeval`. It is a manual production runbook for a self-managed AIB deployment and mirrors the passing wire evaluation. The `researcher` Agent must have a tool that calls the protected resource configured below.

### Production flow

Install KAOS with Keycloak for user and Agent identity and enable the self-managed AIB integration. `--token-exchange-enabled` makes the operator reflect AIB data; it does not create AIB services or permission sets.

```bash .noeval
set -euo pipefail
REPO_ROOT=$(git rev-parse --show-toplevel)
AIB_CHART_PATH=/path/to/agentic-identity-broker/chart

kaos system install --gateway-enabled --metallb-enabled \
  --agent-auth-enabled keycloak --user-auth-enabled keycloak \
  --token-exchange-enabled --aib-chart-path "$AIB_CHART_PATH" \
  --chart-path "$REPO_ROOT/operator/chart" --wait
```

Keycloak 26 must have the `token-exchange` and `admin-fine-grained-authz` features enabled. Create the `token-exchange-broker` target client, allow the `researcher` DCR client to exchange to it, and configure its audience mapper to emit exactly `aud=token-exchange-broker`. A missing feature returns `400 unsupported_grant_type`; a missing per-client permission returns `403 Client not allowed to exchange`.

Set the production service values. The protected-resource hostname is both the Agent-facing hostname matched by the generated route and the generated Backend origin. Production infrastructure must force the Agent's request through the gateway while Envoy resolves the same hostname to the internet.

```bash .noeval
NAMESPACE=token-exchange-demo
AGENT=researcher
PROTECTED_RESOURCE=https://api.github.com/user
THIRD_PARTY_ISSUER=https://github.com
THIRD_PARTY_AUTHORIZE_ENDPOINT=https://github.com/login/oauth/authorize
THIRD_PARTY_TOKEN_ENDPOINT=https://github.com/login/oauth/access_token
THIRD_PARTY_SCOPE=read:user
THIRD_PARTY_CLIENT_ID='replace-with-github-oauth-client-id'
THIRD_PARTY_CLIENT_SECRET='replace-with-github-oauth-client-secret'
```

Administer the service, permission set, and Agent binding through AIB's admin API. The local port-forward and `X-Remote-User` header below match the self-managed evaluation configuration; use authenticated admin access in production.

```bash .noeval
kubectl port-forward -n aib-system svc/aib-agentic-identity-broker 14000:14000 >./tmp/aib-admin-port-forward.log 2>&1 &
AIB_ADMIN=http://localhost:14000/api
ADMIN_HEADER='X-Remote-User: kaos-operator'

SERVICE_ID=$(curl --max-time 60 -fsS -X POST "$AIB_ADMIN/services" \
  -H "$ADMIN_HEADER" -H 'Content-Type: application/json' \
  -d "$(jq -n \
    --arg client_id "$THIRD_PARTY_CLIENT_ID" \
    --arg client_secret "$THIRD_PARTY_CLIENT_SECRET" \
    --arg issuer "$THIRD_PARTY_ISSUER" \
    --arg authorize "$THIRD_PARTY_AUTHORIZE_ENDPOINT" \
    --arg token "$THIRD_PARTY_TOKEN_ENDPOINT" \
    --arg scope "$THIRD_PARTY_SCOPE" \
    --arg resource "$PROTECTED_RESOURCE" \
    '{display_name:"GitHub",client_id:$client_id,client_secret:$client_secret,oauth2_flavor:"github",issuer_uri:$issuer,discovery:{enable_discovery:false},endpoints:{authorize_endpoint:$authorize,token_endpoint:$token},scopes:[{scope_value:$scope,description:"Read the GitHub user profile"}],protected_resources:[$resource]}')" \
  | jq -r .id)

PERMISSION_SET_ID=$(curl --max-time 60 -fsS -X POST "$AIB_ADMIN/permission-sets" \
  -H "$ADMIN_HEADER" -H 'Content-Type: application/json' \
  -d "$(jq -n --arg service_id "$SERVICE_ID" --arg scope "$THIRD_PARTY_SCOPE" \
    '{name:"github-read-user",description:"Read the GitHub user profile",service_scopes:[{service_id:$service_id,scopes:[$scope],requirement_type:"mandatory"}]}')" \
  | jq -r .id)

RESEARCHER_CLIENT_ID=$(kubectl get secret -n "$NAMESPACE" "kaos-oidc-$AGENT" -o jsonpath='{.data.client_id}' | base64 -d)

curl --max-time 60 -fsS -X POST "$AIB_ADMIN/agents" \
  -H "$ADMIN_HEADER" -H 'Content-Type: application/json' \
  -d "$(jq -n \
    --arg client_id "$RESEARCHER_CLIENT_ID" \
    --arg external_id "kaos/$NAMESPACE/$AGENT" \
    --arg permission_set_id "$PERMISSION_SET_ID" \
    '{client_id:$client_id,external_id:$external_id,display_name:"researcher",description:"KAOS exchange-enabled Agent",permission_sets:[{permission_set_id:$permission_set_id,requirement_type:"mandatory"}]}')"
```

Wait for the next reflection pass, which runs every 45 seconds by default. The operator generates the FQDN Backend, HTTPRoute, fail-closed SecurityPolicy, and ext_proc policy from AIB alone, updates the AIB Agent's DCR `client_id`, and injects `KAOS_TOKEN_EXCHANGE_CONFIG` only into the bound Agent. Do not create a Service, Backend, HTTPRoute, ThirdPartyService, or annotation for this integration.

```bash .noeval
sleep 50
kubectl get backend,httproute,securitypolicy,envoyextensionpolicy \
  -n "$NAMESPACE" -l kaos.tools/token-exchange-managed=true

ROUTE_NAME=$(kubectl get httproute -n "$NAMESPACE" \
  -l kaos.tools/token-exchange-managed=true -o jsonpath='{.items[0].metadata.name}')

kubectl get httproute "$ROUTE_NAME" -n "$NAMESPACE" \
  -o jsonpath='{.spec.hostnames[0]}{" ResolvedRefs="}{.status.parents[0].conditions[?(@.type=="ResolvedRefs")].status}{"\n"}'
kubectl get backend "$ROUTE_NAME" -n "$NAMESPACE" \
  -o jsonpath='{.spec.endpoints[0].fqdn.hostname}{":"}{.spec.endpoints[0].fqdn.port}{"\n"}'
kubectl get envoyextensionpolicy "$ROUTE_NAME" -n "$NAMESPACE" \
  -o jsonpath='{.spec.targetRefs[0].kind}{"/"}{.spec.targetRefs[0].name}{" failOpen="}{.spec.extProc[0].failOpen}{"\n"}'

kubectl get deployment "agent-$AGENT" -n "$NAMESPACE" -o json \
  | jq '[.spec.template.spec.containers[0].env[] | select(.name=="KAOS_TOKEN_EXCHANGE_CONFIG")]'
```

Mint the user's normal Keycloak token and call the Agent. The first call without a live AIB vault session returns application HTTP 200 with the controlled `third_party_reauth_required` result and an AIB authorization URL; the third-party request has not reached its Backend.

```bash .noeval
KEYCLOAK_URL=https://keycloak.example.com
GATEWAY_URL=https://gateway.example.com
KAOS_USER_PASSWORD=kaos-password

USER_TOKEN=$(curl --max-time 60 -fsS -X POST \
  "$KEYCLOAK_URL/realms/kaos/protocol/openid-connect/token" \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  --data-urlencode client_id=kaos \
  --data-urlencode client_secret=kaos-dev-secret \
  --data-urlencode username=kaos-user \
  --data-urlencode password="$KAOS_USER_PASSWORD" \
  --data-urlencode grant_type=password | jq -r .access_token)

call_researcher() {
  curl --max-time 60 -fsS "$GATEWAY_URL/$NAMESPACE/agent/$AGENT/v1/chat/completions" \
    -H "Authorization: Bearer $USER_TOKEN" \
    -H 'Content-Type: application/json' \
    -d '{"model":"researcher","messages":[{"role":"user","content":"Call the third-party service"}]}'
}

FIRST_RESULT=$(call_researcher | jq -r '.choices[0].message.content')
echo "$FIRST_RESULT"
# Access to api.github.com requires re-authentication (third_party_reauth_required).
# Please reconnect at https://<aib>/api/third-party/<service-id>/oauth2/authorize and try again.
```

Open the returned URL as the requesting user. Complete the provider's S256 PKCE authorization-code flow and approve the permission in AIB. The redirect finishes at the AIB consent UI, and AIB stores the provider access and refresh tokens in its encrypted vault; KAOS does not store them.

```bash .noeval
REAUTH_URL=$(echo "$FIRST_RESULT" | grep -Eo 'https?://[^ ]+/api/third-party/[^ ]+/oauth2/authorize')
open "$REAUTH_URL"  # use xdg-open on Linux

SUCCESS_RESULT=$(call_researcher | jq -r '.choices[0].message.content')
echo "$SUCCESS_RESULT"
# Third-party tool completed.
```

The successful wire path is `Agent -> gateway -> PDP -> ext_proc -> generated Backend -> third party`. The Agent re-mints the user's Keycloak token before egress; the live evaluation decoded these exact claims, where `azp` is the `researcher` DCR client and `sub` remains the requesting user:

```json .noeval
{
  "aud": "token-exchange-broker",
  "azp": "85be1caf-30b9-4236-87b2-fae29613d86d",
  "sub": "c9df7bbc-c015-4095-b39e-7b5ed1a3f5e9",
  "iss": "http://keycloak.keycloak.svc.cluster.local:8080/realms/kaos"
}
```

The PDP binds that `azp` to the verified Agent actor. AIB ext_proc then swaps the re-minted token for the vaulted provider token, and only the provider token reaches the third party. An unbound Agent receives HTTP 403 before ext_proc. Internal Agent, MCPServer, ModelAPI, and MemoryStore routes retain the original user token and never receive ext_proc.

Revoke the provider session and retry. The evaluation-only pre-auth header below stands in for the authenticated production user. The delete returns HTTP 200, and the next Agent call returns `third_party_reauth_required` again while ext_proc records broker `invalid_grant` and sends nothing upstream.

```bash .noeval
kubectl port-forward -n aib-system svc/aib-agentic-identity-broker 8000:8000 >./tmp/aib-user-port-forward.log 2>&1 &
AIB_URL=http://localhost:8000
USER_SUB=$(printf '%s' "$USER_TOKEN" | cut -d. -f2 | tr '_-' '/+' | base64 -d 2>./tmp/dev/null | jq -r .sub)

curl --max-time 60 -fsS -X DELETE "$AIB_URL/api/third-party/$SERVICE_ID/session" \
  -H "X-Remote-User: $USER_SUB"
# {"message":"session terminated successfully"}

call_researcher | jq -r '.choices[0].message.content'
# Access to api.github.com requires re-authentication (third_party_reauth_required).
```

### KIND-only single-name rig split

This subsection is only for a local mock on KIND. It is not product configuration and must not be copied into production. The mock reuses one in-cluster hostname, so the test pod needs a split view: the Agent resolves the hostname to the gateway LoadBalancer through `hostAliases`, while Envoy's normal CoreDNS resolution sends the generated Backend to the mock Service. The AIB protected resource still contains that one hostname, and there is no alternate-origin annotation.

```bash .noeval
NAMESPACE=token-exchange-demo
MOCK_HOST=mock-api.$NAMESPACE.svc.cluster.local
GATEWAY_IP=$(kubectl get service -n envoy-gateway-system \
  -l gateway.envoyproxy.io/owning-gateway-name=kaos-gateway \
  -o jsonpath='{.items[0].status.loadBalancer.ingress[0].ip}')

# The real mock Service exposes HTTP 80 and forwards to the mock container on 9000.
kubectl patch service mock-api -n "$NAMESPACE" --type=merge \
  -p '{"spec":{"ports":[{"name":"http","port":80,"targetPort":9000}]}}'

# Agent-facing resolution only: the same hostname enters the gateway.
kubectl patch agent researcher -n "$NAMESPACE" --type=merge \
  -p "{\"spec\":{\"podSpec\":{\"containers\":[{\"name\":\"agent\"}],\"hostAliases\":[{\"ip\":\"$GATEWAY_IP\",\"hostnames\":[\"$MOCK_HOST\"]}]}}}"

# Use this value in the AIB service and in the Agent tool.
PROTECTED_RESOURCE=http://$MOCK_HOST/api/data
```

With that KIND-only split, the proved path was `researcher -> gateway -> PDP -> ext_proc -> generated Backend -> mock-api:80`, with `ResolvedRefs=True`, no loop, and no administrator-authored egress Kubernetes object.